# Optimized Fire and Smoke Detection Pipeline

Key Features:

1. Reproducibility
2. Class Balancing
3. SAM Integration
4. Production Readiness

# Environment setup for the detection pipeline.

In [ ]:
import wandb
import warnings
from segment_anything import sam_model_registry, SamPredictor
import cv2
import yaml
from google.colab import drive
import torch
from sklearn.utils.class_weight import compute_class_weight
from collections import defaultdict
from pathlib import Path
import numpy as np
import random
import shutil
import json
import sys
import os
!pip install -q numpy torch torchvision pyyaml albumentations segment-anything
!pip -q install wandb

In [ ]:
# Reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Mount Google Drive
drive.mount('/content/drive')
print(f"GPU: {torch.cuda.get_device_name(0)}")

wandb.init()

## Data processing script for converting COCO annotations to YOLO format.

In [ ]:
class RobustCoco2YoloConverter:
    """Enhanced COCO to YOLO converter with path validation"""

    def __init__(self, coco_path: str, images_root: str, output_root: str, split: str = 'train') -> None:
        self.coco_path = Path(coco_path)
        self.images_root = Path(images_root)
        self.output_root = Path(output_root)
        self.split = split
        self.class_map = {4: 0, 5: 1}  # COCO IDs to YOLO classes
        self.validate_paths()
        self.class_dist: Dict[int, int] = defaultdict(int)

    def validate_paths(self) -> None:
        """Ensure all input paths exist"""
        if not self.coco_path.exists():
            raise FileNotFoundError(
                f"COCO annotations not found at {self.coco_path}")
        if not self.images_root.exists():
            raise FileNotFoundError(
                f"Image directory not found at {self.images_root}")
        self.output_root.mkdir(parents=True, exist_ok=True)

    def convert(self) -> Dict[int, int]:
        """Main conversion logic with error handling"""
        with open(self.coco_path) as f:
            coco_data = json.load(f)

        # Create index mappings
        images = {img['id']: img for img in coco_data['images']}
        annotations: Dict[int, list] = defaultdict(list)

        # Process annotations
        for ann in coco_data['annotations']:
            if ann['category_id'] in self.class_map:
                self.class_dist[self.class_map[ann['category_id']]] += 1
                annotations[ann['image_id']].append(ann)

        # Create YOLO directory structure
        output_dir = self.output_root / self.split
        (output_dir / 'images').mkdir(parents=True, exist_ok=True)
        (output_dir / 'labels').mkdir(parents=True, exist_ok=True)

        # Process images with progress bar
        for img_id, anns in tqdm(annotations.items(), desc=f"Converting {self.split}"):
            img_info = images.get(img_id)
            if not img_info:
                continue

            # Validate source image path
            src_path = self.images_root / img_info['file_name']
            if not src_path.exists():
                print(f"Warning: Missing image {src_path}")
                continue

            # Copy image
            dst_img = output_dir / 'images' / src_path.name
            shutil.copy(src_path, dst_img)

            # Create label file
            label_path = output_dir / 'labels' / f"{src_path.stem}.txt"
            with open(label_path, 'w') as f:
                for ann in anns:
                    # Convert COCO bbox to YOLO format
                    bbox = ann['bbox']
                    img_w, img_h = img_info['width'], img_info['height']

                    x_center = (bbox[0] + bbox[2] / 2) / img_w
                    y_center = (bbox[1] + bbox[3] / 2) / img_h
                    width = bbox[2] / img_w
                    height = bbox[3] / img_h
                    f.write(
                        f"{self.class_map[ann['category_id']]} {x_center} {y_center} {width} {height}\n")

        print(
            f"\nClass Distribution: Smoke={self.class_dist[0]}, Fire={self.class_dist[1]}")
        return dict(self.class_dist)

## Execute conversion with verification

In [ ]:
try:
    train_converter = RobustCoco2YoloConverter(
        '/content/drive/MyDrive/dataset_fire_smoke/475_fire_train/annotations/instances_default.json',
        '/content/drive/MyDrive/dataset_fire_smoke/475_fire_train/images',
        '/content/yolo_dataset',
        'train'
    )
    train_converter.convert()

    val_converter = RobustCoco2YoloConverter(
        '/content/drive/MyDrive/dataset_fire_smoke/474_fire_val/annotations/instances_default.json',
        '/content/drive/MyDrive/dataset_fire_smoke/474_fire_val/images',
        '/content/yolo_dataset',
        'val'
    )
    val_converter.convert()
except Exception as e:
    print(f"\nERROR: {str(e)}")
    sys.exit(1)

## Class balancing for the training dataset.

In [ ]:
classes = np.array([0, 1])
samples = np.concatenate(
    [np.full(train_converter.class_dist[0], 0),
     np.full(train_converter.class_dist[1], 1)]
)
weights = compute_class_weight(
    "balanced", classes=classes, y=samples
)
class_weights: dict[int, float] = {
    0: weights[0].item(),
    1: weights[1].item(),
}

print("Class Weights for Training:")
print(f"Smoke: {class_weights[0]:.2f}, Fire: {class_weights[1]:.2f}")

## Clone YOLOv5 and install requirements

In [ ]:
!git clone -q https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt

## Define hyperparameters

In [ ]:
smoke_weight = round(
    (train_converter.class_dist[1] / train_converter.class_dist[0]), 2)

hyp = {
    'lr0': 0.01,
    'lrf': 0.1,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 0.05,
    'cls': 0.5,
    'cls_pw': smoke_weight,
    'obj': 1.0,
    'obj_pw': 1.0,
    'iou_t': 0.20,
    'anchor_t': 4.0,
    'fl_gamma': 2.0,
    'dfl': 1.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 10.0,
    'translate': 0.1,
    'scale': 0.9,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.0,
    'copy_paste': 0.0,
    'nc': 2,
}

## Save hyperparameters and config to a YAML file

In [ ]:
Path('/content/yolov5/data/hyps').mkdir(parents=True, exist_ok=True)
hyp_file = '/content/yolov5/data/hyps/hyp.fire-smoke.yaml'

with open(hyp_file, 'w') as f:
    yaml.dump(hyp, f, sort_keys=False)

# Create dataset config
with open('data/fire_smoke.yaml', 'w') as f:
    f.write(f"""
train: /content/yolo_dataset/train
val: /content/yolo_dataset/val
nc: 2
names: ['smoke', 'fire']
""")

print(f"YOLO configuration file created successfully at {hyp_file}")

## Training command

In [ ]:
with torch.amp.autocast(device_type='cuda', enabled=True):
    !python train.py --img 640 --batch 16 --epochs 100 \
        --data fire_smoke.yaml --weights yolov5s.pt --cache \
        --device 0 --seed 42 --optimizer AdamW \
        --patience 15 --bbox_interval 10 \
        --hyp data/hyps/hyp.fire-smoke.yaml

## Define class for fire and smoke segmentation

In [ ]:
class FireSegmentation:
    """Combines YOLOv5 detection with SAM segmentation"""

    def __init__(self, yolo_weights, sam_type='vit_b'):
        self.yolo = torch.load(yolo_weights)['model'].fuse()
        self.sam = sam_model_registry[sam_type](
            checkpoint="sam_vit_b_01ec64.pth").to('cuda')
        self.predictor = SamPredictor(self.sam)

    def process(self, img_path):
        # YOLO Detection
        results = self.yolo(img_path)
        detections = results.pandas().xyxy[0].query("confidence > 0.5")

        # SAM Segmentation
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        self.predictor.set_image(img)

        masks = []
        for _, row in detections.iterrows():
            box = row[['xmin', 'ymin', 'xmax', 'ymax']].values.astype(int)
            mask, _, _ = self.predictor.predict(box=box)
            masks.append(mask[0])  # Take best mask

        return detections, masks

## Run validation command to get metrics

In [ ]:
!python val.py --device 0 --data fire_smoke.yaml \
    --weights runs/train/exp/weights/best.pt \
    --img 640 --batch-size 32 \
    --name final_validation \
    --save-json --save-conf --save-txt

## ONNX Export

In [ ]:
!python export.py --weights runs/train/exp/weights/best.pt --include onnx

## Save to Drive

In [ ]:
!cp runs/train/exp/weights/best.onnx /content/drive/MyDrive/fire_smoke_model.onnx
!zip -qr /content/yolo_dataset.zip /content/yolo_dataset
!cp /content/yolo_dataset.zip /content/drive/MyDrive/